# Maia Lite 0.3.0 — Corpus experimental de escala (200 mil)
**Objetivo:** coletar até 200.000 diálogos reais de fontes públicas selecionadas, sem aprovação manual por lote, para **experimento privado**. Não certifica licenças, qualidade ou adequação comercial. Nunca inventa registros para atingir a meta. **Este notebook não treina o modelo.**

Os dados são registrados com origem e revisão. `target_total` é uma meta, não garantia. A construção pode terminar abaixo da meta se as fontes não fornecerem dados suficientes. Não publicar corpus ou pesos antes da revisão das licenças e direitos sobre dados derivados.

## 1. Dependências mínimas — preservar o ambiente Colab

In [ ]:
import importlib.util,subprocess,sys
missing=[p for p in ('datasets','huggingface_hub','langdetect','tqdm') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])


## 2. Configuração

In [ ]:
from pathlib import Path
import json,os,re,hashlib,random,time,collections,unicodedata
from datasets import load_dataset
from huggingface_hub import HfApi
from langdetect import detect,DetectorFactory
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')
SEED=20261120; DetectorFactory.seed=SEED
TARGET=200_000; MAX_ROWS_PER_SOURCE=500_000; MAX_TURNS=12
MIN_CHARS=12; MAX_CHARS_PER_MESSAGE=4500
RUN_FULL=True  # False: diagnóstico com limite reduzido
SOURCE_LIMIT=MAX_ROWS_PER_SOURCE if RUN_FULL else 200
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
OUT=ROOT/'datasets/maia_conversational_experiment_200k_v1'
OUT.mkdir(parents=True,exist_ok=True)
print('Destino:',OUT)


## 3. Fontes
OASST2 (Apache-2.0 declarado), UltraChatBR (MIT declarado), Aya (Apache-2.0 declarado). **Declarações do cartão não são verificação de licenças de conteúdos de origem**. No Robots e Canarim (CC BY-NC) ficam excluídos. O código tenta várias configurações e registra erros sem bloquear outras fontes.

In [ ]:
SOURCES=[
 {'id':'oasst2','repo':'OpenAssistant/oasst2','kind':'tree','split':'train','license_note':'apache-2.0 declared'},
 {'id':'ultrachatbr','repo':'recogna-nlp/UltrachatBR','kind':'chat','split':'train','license_note':'mit declared; upstream unverified'},
 {'id':'aya','repo':'CohereLabs/aya_dataset','kind':'pair','split':'train','license_note':'apache-2.0 declared'},
]
api=HfApi();manifest=[]
for s in SOURCES:
    item=dict(s)
    try:
        info=api.dataset_info(s['repo']);item['revision']=info.sha;item['card_license']=getattr(info.cardData,'license',None) if info.cardData else None
    except Exception as e:item['metadata_error']=str(e)[:300];item['revision']=None
    manifest.append(item)
print(json.dumps(manifest,ensure_ascii=False,indent=2))


## 4. Normalização e ingestão (sem aprovação manual)

In [ ]:
def as_text(v):
    if isinstance(v,str):return v.strip()
    if isinstance(v,dict):return str(v.get('text') or v.get('content') or '').strip()
    return ''
def clean_messages(seq):
    ans=[]
    for m in seq:
        if not isinstance(m,dict):continue
        role=str(m.get('role') or m.get('from') or m.get('speaker') or '').lower()
        role={'human':'user','prompter':'user','gpt':'assistant','bot':'assistant'}.get(role,role)
        txt=as_text(m.get('content') or m.get('text') or m.get('value'))
        if role not in ('user','assistant') or len(txt)<MIN_CHARS or len(txt)>MAX_CHARS_PER_MESSAGE:continue
        if ans and ans[-1]['role']==role:continue
        ans.append({'role':role,'content':txt})
    while ans and ans[0]['role']!='user':ans.pop(0)
    if len(ans)%2:ans.pop()
    return ans[:MAX_TURNS] if len(ans)>=2 else []
def normalize_row(row,kind):
    if kind=='pair':
        q=as_text(row.get('inputs') or row.get('instruction') or row.get('prompt') or row.get('question'))
        a=as_text(row.get('targets') or row.get('output') or row.get('response') or row.get('answer'))
        return clean_messages([{'role':'user','content':q},{'role':'assistant','content':a}])
    for key in ('messages','conversations','conversation','chat'):
        val=row.get(key)
        if isinstance(val,list):return clean_messages(val)
    return normalize_row(row,'pair') if kind=='chat' else []
def language(row,ms):
    v=str(row.get('lang') or row.get('language') or row.get('language_code') or '').lower()
    v={'eng':'en','english':'en','por':'pt','portuguese':'pt','spa':'es','spanish':'es','pt-br':'pt','pt_br':'pt'}.get(v,v[:2])
    if v in ('en','pt','es'):return v
    sample=' '.join(x['content'][:350] for x in ms[:4])
    try:
        v=detect(sample)
        return v if v in ('en','pt','es') else None
    except Exception:return None
def key_of(ms):
    s='\n'.join(m['role']+':'+unicodedata.normalize('NFKC',m['content']).casefold().strip() for m in ms)
    return hashlib.sha256(s.encode()).hexdigest()
def open_stream(s):
    return load_dataset(s['repo'],split=s['split'],streaming=True,revision=s.get('revision'))


## 5. Coleta incremental e checkpoint de progresso
OASST2 exige reconstrução por `parent_id`. As outras fontes usam streaming. Dados são gravados imediatamente em `candidates.jsonl`; em caso de nova execução, o arquivo é reconstruído deterministicamente. **Não use a mesma pasta em execuções concorrentes.**

In [ ]:
from collections import defaultdict,Counter
CAND=OUT/'candidates.jsonl';errors=[];stats=Counter();seen=set();counts=Counter()
def emit(f,s,ms,lang,origin):
    if not ms or lang not in ('en','pt','es'):return False
    fp=key_of(ms)
    if fp in seen:return False
    seen.add(fp)
    record={'id':'maia_'+fp[:24],'source_id':s['id'],'source_repo':s['repo'],
            'source_revision':s.get('revision'),'lang':lang,'category':'unclassified',
            'messages':ms,'origin_id':str(origin)[:200]}
    f.write(json.dumps(record,ensure_ascii=False)+'\n');counts[(s['id'],lang)]+=1
    return True

def ingest_tree(s,f):
    ds=load_dataset(s['repo'],split=s['split'],revision=s.get('revision'))
    nodes={};children=defaultdict(list)
    for row in ds:
        mid=row.get('message_id');parent=row.get('parent_id')
        if mid is None:continue
        nodes[mid]=row
        if parent:children[parent].append(mid)
    emitted=0
    for mid,row in nodes.items():
        if row.get('role')!='assistant' or not row.get('parent_id'):continue
        path=[];cur=mid;visited=set()
        while cur in nodes and cur not in visited and len(path)<MAX_TURNS:
            visited.add(cur);r=nodes[cur];role={'prompter':'user','assistant':'assistant'}.get(r.get('role'))
            if role:
                txt=as_text(r.get('text'))
                path.append({'role':role,'content':txt})
            cur=r.get('parent_id')
        ms=clean_messages(list(reversed(path)))
        if emit(f,s,ms,language(row,ms),mid):emitted+=1
        if emitted>=SOURCE_LIMIT:break
    return emitted

def ingest_stream(s,f):
    it=open_stream(s);emitted=0
    for idx,row in enumerate(tqdm(it,desc=s['id'],total=SOURCE_LIMIT)):
        if idx>=SOURCE_LIMIT or sum(counts.values())>=TARGET:break
        ms=normalize_row(row,s['kind'])
        if emit(f,s,ms,language(row,ms),row.get('id',idx)):emitted+=1
    return emitted

with CAND.open('w',encoding='utf-8') as f:
    for s in manifest:
        if sum(counts.values())>=TARGET:break
        try:
            count=ingest_tree(s,f) if s['kind']=='tree' else ingest_stream(s,f)
            print(s['id'],'adicionados:',count)
        except Exception as e:
            errors.append({'source':s['id'],'error':repr(e)});print('FALHA:',s['id'],repr(e))
        f.flush()
print('Total candidatos:',sum(counts.values()),'erros:',errors)


## 6. Seleção e divisão experimental
Seleciona no máximo 200 mil candidatos, sem forçar cotas impossíveis. **Separação por hash da conversa**; não é garantia de ausência de vazamento semântico. Uma pequena divisão de teste fica disponível, mas não há revisão humana nem bloqueio de treino por cobertura.

In [ ]:
import csv
rng=random.Random(SEED)
rows=[]
with CAND.open(encoding='utf-8') as f:
    for line in f:
        if line.strip():rows.append(json.loads(line))
rng.shuffle(rows)
rows=rows[:TARGET]
# 1% teste; nenhuma validação periódica é obrigatória para este experimento.
test_n=max(1,round(len(rows)*0.01)) if len(rows)>=100 else 0
splits={'train':rows[test_n:],'test':rows[:test_n]}
for split,items in splits.items():
    with (OUT/(split+'.jsonl')).open('w',encoding='utf-8') as f:
        for r in items:f.write(json.dumps(r,ensure_ascii=False)+'\n')
with (OUT/'sources_manifest.json').open('w') as f:json.dump(manifest,f,indent=2,ensure_ascii=False)
with (OUT/'ingest_errors.json').open('w') as f:json.dump(errors,f,indent=2,ensure_ascii=False)
with (OUT/'coverage.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['source','lang','count'])
    for (src,lang),v in sorted(counts.items()):w.writerow([src,lang,v])
report={'target':TARGET,'candidates':len(rows),'train':len(splits['train']),
        'test':len(splits['test']),'source_counts':{f'{s}:{l}':v for (s,l),v in counts.items()},
        'ingest_errors':errors,'license_audited':False,'quality_audited':False,
        'experimental_only':True,'status':'EXPERIMENTAL_NOT_RELEASE_APPROVED'}
(OUT/'build_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
for p in sorted(OUT.glob('*')):
    if p.is_file():print(p.name,round(p.stat().st_size/1e6,2),'MB')
print(json.dumps(report,ensure_ascii=False,indent=2))


## 7. Próximo passo
Abra o notebook **Maia_Lite_0_3_0_Experimental_SFT_200k_L4.ipynb**. Ele lê `train.jsonl`, converte conversas em pares de turnos, aplica máscara de loss só no assistente e inicia LoRA em novo diretório. Se a coleta ficar muito abaixo de 200 mil, não suponha que a meta foi cumprida: veja `build_report.json` e `ingest_errors.json`.